# Pipeline A (v5): Fine-Tuning on Real Phone Recordings

Fine-tunes the existing SpoofCNN (v4) on collected real-world bonafide recordings to reduce false positives. 

**New bonafide data:** 26 recordings from 10 speakers (Sri  Lankan L2 English, various ages/genders)
**Spoof data:** Subset of ASVspoof2019 LA training spoofs (same as original training)
**Strategy:** Low learning rate fine-tuning to shift the decision boundary without major forgetting.

## Setup & Reproducability

In [1]:
# ---- Imports----
import os
import random
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, confusion_matrix
import matplotlib.pyplot as plt


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Configuration

In [2]:
# ---- Config -----
SAMPLE_RATE = 16000
CLIP_SECONDS = 4.0
TARGET_LEN   = int(SAMPLE_RATE * CLIP_SECONDS)
N_MELS, N_FFT, HOP_LENGTH, WIN_LENGTH = 64, 1024, 160, 400
F_MIN, F_MAX = 20, 7600

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


# Paths
BASE            = Path(r"C:\FYP")
COLLECTED_DIR   = BASE / "Datasets" / "collected_bonafide"
PB_TESTING_DIR  = Path(r"C:\Users\User\Desktop\PB testing audio")
ASVSPOOF_DIR    = BASE / "Datasets" / "ASVspoof2019_LA"
PROTOCOL_FILE   = ASVSPOOF_DIR / "ASVspoof2019_LA_cm_protocols" / "ASVspoof2019.LA.cm.train.trn.txt"
TRAIN_AUDIO_DIR = ASVSPOOF_DIR / "ASVspoof2019_LA_train" / "flac"
ORIG_MODEL_PATH = BASE / "fyp-speaking-skills-app" / "Backend" / "models" / "best_spoof_cnn_v4.pth"
SAVE_PATH       = BASE / "fyp-speaking-skills-app" / "Backend" / "models" / "best_spoof_cnn_v5.pth"

#Fine-tuning hyperparameters
LEARNING_RATE  = 1e-4   # Low LR to avoid forgetting original knowledge
EPOCHS         = 10
BATCH_SIZE     = 16
SPOOF_SAMPLES  = 300    # Number of ASVspoof spoof samples to include
SEED           = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("Config loaded.")



Device: cuda
Config loaded.


In [3]:
LABEL_PATH = Path("C:/FYP/Datasets/ASVspoof2019_LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt")
SPOOF_DIR = Path("C:/FYP/Datasets/ASVspoof2019_LA/ASVspoof2019_LA_train")

# Model Architecture

In [4]:
# ===== CNN Architecture =====
class SpoofCNN(nn.Module):
    def __init__(self, n_mels=64):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.BatchNorm2d(16),
            nn.LeakyReLU(0.1), nn.MaxPool2d(2), nn.Dropout(0.2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32),
            nn.LeakyReLU(0.1), nn.MaxPool2d(2), nn.Dropout(0.3),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64),
            nn.LeakyReLU(0.1), nn.MaxPool2d(2), nn.Dropout(0.4),
        )
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64, 64), nn.LeakyReLU(0.1),
            nn.Dropout(0.4), nn.Linear(64, 2)
        )
    def forward(self, x):
        return self.classifier(self.gap(self.features(x)))

class LogMelExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.melspec = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE, n_fft=N_FFT, win_length=WIN_LENGTH,
            hop_length=HOP_LENGTH, f_min=F_MIN, f_max=F_MAX,
            n_mels=N_MELS, power=2.0)
        self.amp_to_db = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=80)
    def forward(self, wav):
        mel = self.melspec(wav)
        logmel = self.amp_to_db(mel)
        return (logmel - logmel.mean()) / logmel.std().clamp_min(1e-6)

extractor = LogMelExtractor().to(DEVICE)
print("Model architecture defined.")

Model architecture defined.


## Audio Loading

In [5]:
def load_wav(path: Path) -> torch.Tensor:
    '''Load WAV file, force mono, resample to 16kHz, fix length to 4 seconds.'''
    wav_np, sr = sf.read(str(path), dtype="float32", always_2d=True)
    wav = torch.from_numpy(wav_np).transpose(0, 1)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != SAMPLE_RATE:
        wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
    # Fix length to exactly 4 seconds
    t = wav.shape[-1]
    if t > TARGET_LEN:
        wav = wav[..., :TARGET_LEN]
    elif t < TARGET_LEN:
        wav = F.pad(wav, (0, TARGET_LEN - t))
    return wav

# Test it
test_file = list(COLLECTED_DIR.glob("*.wav"))[0]
test_wav = load_wav(test_file)
print(f"Test load: {test_file.name} → shape {test_wav.shape}")

Test load: DR2004_P3.wav → shape torch.Size([1, 64000])


# Build Dataset:

## Helper

In [6]:
VALID_AUDIO_EXTS = {".wav", ".mp3", ".m4a", ".flac"}

def is_valid_audio(path: Path) -> bool:
    try:
        wav_np, sr = sf.read(str(path), dtype="float32", always_2d=True)

        if wav_np is None or wav_np.size == 0:
            raise ValueError("Empty audio")

        return True

    except Exception as e:
        print(f"Skipping invalid file: {path.name} | {e}")
        return False

## Build dataset

In [7]:
# ==== Build mixed fine-tuning dataset ====

bonafide_files = sorted([
    p for p in COLLECTED_DIR.glob("*")
    if p.is_file() and p.suffix.lower() in VALID_AUDIO_EXTS
])

# Keep only files that can actually be loaded
bonafide_files = [p for p in bonafide_files if is_valid_audio(p)]

spoof_files = []
with open(LABEL_PATH, "r") as f:
    for line in f:
        parts = line.strip().split()
        file_id = parts[1]
        label = parts[4].lower()

        if label == "spoof":
            spoof_path = SPOOF_DIR / "flac" / f"{file_id}.flac"
            if spoof_path.exists():
                spoof_files.append(spoof_path)

# limit spoof samples to reduce imbalance
spoof_files = sorted(spoof_files)[:SPOOF_SAMPLES]

all_samples = [(p, 0) for p in bonafide_files] + [(p, 1) for p in spoof_files]

print(f"Total bonafide samples: {len(bonafide_files)}")
print(f"Total spoof samples: {len(spoof_files)}")
print(f"Total samples: {len(all_samples)}")
print(f"Class balance — Bonafide: {len(bonafide_files)}, Spoof: {len(spoof_files)}")

Total bonafide samples: 32
Total spoof samples: 300
Total samples: 332
Class balance — Bonafide: 32, Spoof: 300


In [8]:
class FineTuneDataset(Dataset):
    def __init__(self, samples, target_len=TARGET_LEN):
        self.samples = samples
        self.target_len = target_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        try:
            wav_np, sr = sf.read(str(path), dtype="float32", always_2d=True)
            wav = torch.from_numpy(wav_np.T)  # [channels, time]

        except Exception as e:
            raise RuntimeError(f"Failed to load {path}: {e}")

        # force mono
        if wav.size(0) > 1:
            wav = wav.mean(dim=0, keepdim=True)

        # resample if needed
        if sr != SAMPLE_RATE:
            wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)

        # fix length to exactly TARGET_LEN
        if wav.size(1) > self.target_len:
            wav = wav[:, :self.target_len]
        elif wav.size(1) < self.target_len:
            pad = self.target_len - wav.size(1)
            wav = F.pad(wav, (0, pad))

        return wav, label

# Dataset Class and split

In [9]:
from sklearn.model_selection import train_test_split

labels = [label for _, label in all_samples]

train_samples, val_samples = train_test_split(
    all_samples,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

train_ds = FineTuneDataset(train_samples)
val_ds   = FineTuneDataset(val_samples)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")
print("Train bonafide:", sum(label == 0 for _, label in train_samples))
print("Train spoof:",    sum(label == 1 for _, label in train_samples))
print("Val bonafide:",   sum(label == 0 for _, label in val_samples))
print("Val spoof:",      sum(label == 1 for _, label in val_samples))

Train: 265, Val: 67
Train bonafide: 26
Train spoof: 239
Val bonafide: 6
Val spoof: 61


# LOad pretrained model:

In [10]:
model = SpoofCNN(N_MELS).to(DEVICE)
ckpt  = torch.load(ORIG_MODEL_PATH, map_location=DEVICE, weights_only=True)
model.load_state_dict(ckpt["model_state"])
print("Loaded pretrained v4 weights.")

# Use a lower learning rate for the classifier head, even lower for feature layers
# This preserves the learned features while adapting the decision boundary
optimizer = torch.optim.Adam([
    {"params": model.features.parameters(), "lr": LEARNING_RATE * 0.1},
    {"params": model.gap.parameters(),      "lr": LEARNING_RATE * 0.1},
    {"params": model.classifier.parameters(),"lr": LEARNING_RATE},
], lr=LEARNING_RATE)

criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2
)
print("Optimizer ready.")

Loaded pretrained v4 weights.
Optimizer ready.


# Training Loop:

In [11]:
best_val_f1   = 0.0
train_losses  = []
val_f1s       = []

for epoch in range(1, EPOCHS + 1):
    # ==== Train =====
    model.train()
    epoch_loss = 0.0

    for batch_idx, (wavs, labels) in enumerate(train_loader):
        wavs = wavs.squeeze(1).to(DEVICE)   # [B, T]
        labels = labels.to(DEVICE)

        with torch.no_grad():
            feats = extractor(wavs).unsqueeze(1)   # [B, 1, 64, time]

        if epoch == 1 and batch_idx == 0:
            print("TRAIN wavs shape:", wavs.shape)
            print("TRAIN feats shape:", feats.shape)

        optimizer.zero_grad()
        logits = model(feats)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)

    #=== Validate ====
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch_idx, (wavs, labels) in enumerate(val_loader):
            wavs = wavs.squeeze(1).to(DEVICE)      # [B, T]
            feats = extractor(wavs).unsqueeze(1)   # [B, 1, 64, time]

            if epoch == 1 and batch_idx == 0:
                print("VAL wavs shape:", wavs.shape)
                print("VAL feats shape:", feats.shape)

            preds = model(feats).argmax(dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    val_f1 = f1_score(all_labels, all_preds, average="macro")
    val_f1s.append(val_f1)
    scheduler.step(val_f1)

    print(f"Epoch {epoch:02d} | Loss: {avg_loss:.4f} | Val F1: {val_f1:.4f}"
          + (" ← best" if val_f1 > best_val_f1 else ""))

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(
            {"model_state": model.state_dict(), "epoch": epoch, "val_f1": val_f1},
            SAVE_PATH
        )

print(f"\nBest Val F1: {best_val_f1:.4f}")

TRAIN wavs shape: torch.Size([16, 64000])
TRAIN feats shape: torch.Size([16, 1, 64, 401])
VAL wavs shape: torch.Size([16, 64000])
VAL feats shape: torch.Size([16, 1, 64, 401])
Epoch 01 | Loss: 0.4178 | Val F1: 0.7839 ← best
Epoch 02 | Loss: 0.3184 | Val F1: 0.7341
Epoch 03 | Loss: 0.1996 | Val F1: 0.7341
Epoch 04 | Loss: 0.1836 | Val F1: 0.7341
Epoch 05 | Loss: 0.1748 | Val F1: 0.7341
Epoch 06 | Loss: 0.1410 | Val F1: 0.7341
Epoch 07 | Loss: 0.1409 | Val F1: 0.7341
Epoch 08 | Loss: 0.2037 | Val F1: 0.7341
Epoch 09 | Loss: 0.1799 | Val F1: 0.7341
Epoch 10 | Loss: 0.1629 | Val F1: 0.7341

Best Val F1: 0.7839


# Evaluate and show confusion matrix

In [12]:
# Load best checkpoint
ckpt = torch.load(SAVE_PATH, map_location=DEVICE, weights_only=True)
model.load_state_dict(ckpt["model_state"])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for batch_idx, (wavs, labels) in enumerate(val_loader):
        wavs = wavs.squeeze(1).to(DEVICE)          # [B, T]
        feats = extractor(wavs).unsqueeze(1)       # [B, 1, 64, time]

        if batch_idx == 0:
            print("EVAL wavs shape:", wavs.shape)
            print("EVAL feats shape:", feats.shape)

        preds = model(feats).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
print("Confusion Matrix (rows=true, cols=pred):")
print(f"           Pred Bonafide  Pred Spoof")
print(f"True Bonafide    {cm[0][0]:>6}      {cm[0][1]:>6}")
print(f"True Spoof       {cm[1][0]:>6}      {cm[1][1]:>6}")

bonafide_recall = cm[0][0] / cm[0].sum() if cm[0].sum() > 0 else 0.0
spoof_recall    = cm[1][1] / cm[1].sum() if cm[1].sum() > 0 else 0.0
print(f"\nBonafide recall (how many real voices pass): {bonafide_recall:.1%}")
print(f"Spoof recall (how many TTS are caught):      {spoof_recall:.1%}")

EVAL wavs shape: torch.Size([16, 64000])
EVAL feats shape: torch.Size([16, 1, 64, 401])
Confusion Matrix (rows=true, cols=pred):
           Pred Bonafide  Pred Spoof
True Bonafide         3           3
True Spoof            1          60

Bonafide recall (how many real voices pass): 50.0%
Spoof recall (how many TTS are caught):      98.4%
